In [1]:
import os
import gradio as gr
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv 

c:\Users\Admin\Desktop\Nezami_rag_system\NEZAMY_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin\AppData\Local\Temp\ipykernel_31844\3901948532.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [2]:
load_dotenv()
DB_DIRECTORY = "./vectorDB_V4"

In [3]:
def initialize_retrievers(db_path: str):
    """
    Loads the Vector Database and initializes the BM25 Keyword Retriever.
    """
    print("Loading retrieval models...")
    embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
    vector_db = Chroma(persist_directory=db_path, embedding_function=embedding_model)
    return vector_db

In [12]:
def initialize_llm_chain():
    """
    Initializes the Gemini model and the RAG Prompt Template.
    """
    llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)
    system_instruction = """
    You are a helpful and friendly assistant designed to help people easily understand the Saudi Labor Law.

    Your task is to answer the user's question based ONLY on the provided retrieved context.

    Strict Rules:
    1. Answer accurately and concisely using ONLY the provided context.
    2. Always answer in Arabic.
    3. Do not use external knowledge or assume facts that are not provided in the retrieved context or the user's question.
    4. If the relevant legal rule is found in the retrieved context, but the user's question is missing information required to determine the final answer, clearly state what information is missing and explain why it is required.
    5. If the retrieved context does not contain a relevant legal rule for the user's question, reply exactly with:
    "عذراً، لا توجد معلومات كافية في نظام العمل للإجابة على هذا السؤال."
    6. Answer naturally and directly.
    7. When possible, mention the relevant article based on the retrieved context.

    Retrieved Context:
    {context}
    """
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        ("human", "Question: {question}")
    ])
    return prompt_template | llm | StrOutputParser()

In [13]:
VECTOR_DB = initialize_retrievers(DB_DIRECTORY)
GENERATION_CHAIN = initialize_llm_chain()

Loading retrieval models...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13493.27it/s]


In [14]:
def perform_hybrid_search(query: str):
    """
    Executes a hybrid search (Vector + BM25) and formats the results.
    """
    chroma_res = VECTOR_DB.similarity_search(query, k=2)
    unique_docs = list({doc.page_content: doc for doc in chroma_res}.values())
    
    if not unique_docs:
        return "No matching legal articles found.", ""
        
    formatted_sources = ""
    for i, doc in enumerate(unique_docs[:2]):
        formatted_sources += f"Result #{i+1}:\n{doc.page_content.strip()}\n"
        formatted_sources += "=" * 40 + "\n\n"
    
    context_text = "\n\n".join([doc.page_content for doc in unique_docs])
    return formatted_sources, context_text

In [15]:
def generate_ai_answer(query: str):
    """
    Generates the final AI response based on the retrieved context.
    """
    formatted_sources, context_text = perform_hybrid_search(query)
    
    if not context_text:
        return "عذراً، لا توجد معلومات كافية في نظام العمل للإجابة على هذا السؤال.", formatted_sources
        
    answer = GENERATION_CHAIN.invoke({
        "context": context_text,
        "question": query
    })
    return answer, formatted_sources

In [16]:
def launch_gradio_ui():
    """
    Builds and launches the interactive Gradio Web UI with RTL support.
    """

    with gr.Blocks(theme=gr.themes.Soft(), css=".rtl { direction: rtl; text-align: right; }") as demo:
        

        gr.Markdown("# ⚖️ المستشار الذكي لنظام العمل السعودي", elem_classes="rtl")
        
        with gr.Tabs(elem_classes="rtl"):

            with gr.Tab("المستشار الذكي (AI)", elem_classes="rtl"):
                with gr.Row():
                    with gr.Column(scale=2):
                        ai_input = gr.Textbox(label="اسأل المستشار:", placeholder="مثال: هل يجوز تمديد فترة التجربة؟")
                        ai_btn = gr.Button("إرسال", variant="primary")
                    with gr.Column(scale=3):
                        ai_output = gr.Textbox(label="الإجابة:", lines=4, interactive=False)
                        ai_sources = gr.Textbox(label="المصادر المسترجعة:", lines=6, interactive=False)
                ai_btn.click(fn=generate_ai_answer, inputs=ai_input, outputs=[ai_output, ai_sources])
                

            with gr.Tab("محرك البحث المباشر", elem_classes="rtl"):
                search_input = gr.Textbox(label="كلمة البحث:")
                search_btn = gr.Button("ابحث في المواد", variant="primary")
                search_output = gr.Textbox(label="المواد المسترجعة:", lines=15, interactive=False)
                search_btn.click(fn=lambda q: perform_hybrid_search(q)[0], inputs=search_input, outputs=search_output)

    print("Launching the Gradio Interface...")
    demo.launch(inbrowser=True)

In [17]:
launch_gradio_ui()

C:\Users\Admin\AppData\Local\Temp\ipykernel_31844\2815899366.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=".rtl { direction: rtl; text-align: right; }") as demo:


Launching the Gradio Interface...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
